# WER on `train_all`

Corpus and per-region WER for the dialect-aware (FHNW STT4SG) and dialect-ignorant (Whisper-large-v2) hypotheses on the `train_all` split. No text normalization &mdash; raw strings into `jiwer.wer`. Evaluated on the path-intersection of clips flagged `clip_is_usable=True` in the Whisper run so both models score the same audio.

In [1]:
import pandas as pd
from jiwer import wer

DAT_TSV = "../../transcripts/dialect-aware/fhnw/stt4sg/train_all_transcribed.tsv"
DIT_TSV = "../../transcripts/dialect-ignorant/whisper-large-v2/stt4sg/train_all_enriched_transcribed_praet.tsv"

In [2]:
dat = pd.read_csv(DAT_TSV, sep="\t", encoding="utf-8-sig")[
    ["path", "dialect_region", "sentence", "fhnw_transcript"]
]
dit = pd.read_csv(DIT_TSV, sep="\t", encoding="utf-8-sig")[
    ["path", "clip_is_usable", "whisper_large_v2_transcript"]
]

dit = dit[dit["clip_is_usable"].fillna(False).astype(bool)]

df = dat.merge(dit.drop(columns="clip_is_usable"), on="path", how="inner")
df = df[
    df["sentence"].fillna("").str.strip().ne("")
    & df["fhnw_transcript"].fillna("").str.strip().ne("")
    & df["whisper_large_v2_transcript"].fillna("").str.strip().ne("")
].reset_index(drop=True)

print(f"Evaluated clips: {len(df):,}")
print(f"Regions:         {df['dialect_region'].nunique()}")

Evaluated clips: 199,456
Regions:         7


In [3]:
refs = df["sentence"].tolist()
dat_wer = wer(refs, df["fhnw_transcript"].tolist())
dit_wer = wer(refs, df["whisper_large_v2_transcript"].tolist())

print(f"DAT (FHNW)              WER: {dat_wer:.4f}")
print(f"DIT (Whisper-large-v2)  WER: {dit_wer:.4f}")
print(f"Δ (DIT − DAT):              {dit_wer - dat_wer:+.4f}")

DAT (FHNW)              WER: 0.1561
DIT (Whisper-large-v2)  WER: 0.2686
Δ (DIT − DAT):              +0.1126


In [4]:
REGION_ORDER = ["Wallis", "Zürich", "Bern", "Basel", "Graubünden", "Innerschweiz", "Ostschweiz"]

rows = []
for region in REGION_ORDER:
    sub = df[df["dialect_region"] == region]
    if sub.empty:
        continue
    rows.append({
        "region": region,
        "n": len(sub),
        "DAT WER": wer(sub["sentence"].tolist(), sub["fhnw_transcript"].tolist()),
        "DIT WER": wer(sub["sentence"].tolist(), sub["whisper_large_v2_transcript"].tolist()),
    })
per_region = pd.DataFrame(rows)
per_region["Δ (DIT − DAT)"] = per_region["DIT WER"] - per_region["DAT WER"]
per_region

,region,n,DAT WER,DIT WER,Δ (DIT − DAT)
0,Wallis,29606,0.179954,0.374666,0.194712
1,Zürich,28869,0.149366,0.239188,0.089822
2,Bern,28721,0.156382,0.251706,0.095324
3,Basel,27291,0.139876,0.244988,0.105113
4,Graubünden,24043,0.156710,0.262973,0.106263
5,Innerschweiz,29579,0.149184,0.226146,0.076963
6,Ostschweiz,31347,0.159430,0.275992,0.116562
